In [1]:
from pydantic import BaseModel, Field, conint, confloat, ConfigDict
from enum import Enum
from typing import Optional, List
from outlines import models, generate
from pydantic import ValidationError
import json
import pandas as pd
from torch.cuda import empty_cache
import time

empty_cache()

class SexoJuiz(str, Enum):
    MASCULINO = "Masculino"
    FEMININO = "Feminino"

class SimNao(str, Enum):
    SIM = "Sim"
    NAO = "Não"

class RegimeInicial(str, Enum):
    FECHADO = "Fechado"
    SEMIABERTO = "Semi-aberto"
    ABERTO = "Aberto"
    NONE = "None"

class SentencaModel(BaseModel):
    model_config = ConfigDict(use_enum_values=True)  # Usa valores de Enum em vez de objetos

    processo: str = Field(..., pattern=r'^\d{20}$')
    juiz: str
    sexo_juiz: SexoJuiz
    vara: str
    nome: str
    local: str
    maconha: Optional[str] = "None"
    maconha_g: Optional[confloat(ge=0)] = 0
    cocaina: Optional[str] = "None"
    cocaina_g: Optional[confloat(ge=0)] = 0
    crack: Optional[str] = "None"
    crack_g: Optional[confloat(ge=0)] = 0
    ecstasy: Optional[str] = "None"
    ecstasy_g: Optional[confloat(ge=0)] = 0
    lsd: Optional[str] = "None"
    lsd_g: Optional[confloat(ge=0)] = 0
    outras: Optional[str] = "None"
    anabolizantes: SimNao
    anorexigenos: SimNao
    haxixe: SimNao
    skank: SimNao
    lanca_perfume: SimNao
    tolueno: SimNao
    den_drog: str
    den_outros: Optional[str] = "None"
    sentenca: str
    res_drogas: str
    res_outros: Optional[str] = "None"
    pena_base: str
    agravantes33_agrup: SimNao
    confissao: SimNao
    menoridade: SimNao
    atenuantes33_agrup: SimNao
    adolescente: SimNao
    arma_de_fogo: SimNao
    interestadual: SimNao
    concurso_formal: SimNao
    estabelecimento: SimNao
    aumento33_agrup: SimNao
    paragrafo_4o_agrupado: str
    pena33: str
    pena33_meses: conint(ge=0)
    pena_drogas: str
    pena_outros: Optional[str] = "None"
    tot_pen: str
    tot_pen_meses: conint(ge=0)
    substituicao_da_pena: Optional[str] = "None"
    regime_inicial: RegimeInicial
    flag_local_de_trafico: bool = False
    flag_preso_no_momento_da_sentenca: bool = False
    flag_confissao_informal: bool = False
    flag_confissao: bool = False
    flag_denuncia_anonima: bool = False
    flag_denuncia: bool = False
    flag_atitude_suspeita: bool = False
    flag_divergencias_nos_relatos_dos_policiais: bool = False
    flag_investigacao: bool = False
    flag_interceptacao: bool = False
    flag_mandado: bool = False
    flag_nacionalidade: bool = False
    flag_revista_vexatoria: bool = False
    aval_antecedentes: bool = False
    aval_conduta: bool = False
    aval_personalidade: bool = False
    aval_natureza: bool = False
    aval_quantidade: bool = False
    aval_variedade: bool = False
    aval_circunstancias: bool = False
    aval_consequencias: bool = False
    aval_culpabilidade: bool = False

In [2]:
schema_json = json.dumps(SentencaModel.model_json_schema(), indent=2, ensure_ascii=False)
#print(schema_json)

In [3]:
PROMPT_TEMPLATE = """\
[INST] <<SYS>>
Objetivo:
Você deverá extrair informações de sentenças judiciais de processos criminais brasileiros, com foco em delitos relacionados a tráfico de drogas e correlatos, preenchendo os dados de um dataset estruturado com diversos campos. Cada resposta deve ser produzida para cada par processo/reu – isto é, se na sentença houver mais de um réu, gere um objeto JSON separado para cada par, mantendo o mesmo número de processo para todos, mas com as informações específicas de cada réu.

Contexto:
Você receberá a íntegra de uma sentença judicial (ou de uma ata de audiência contendo a sentença) de um processo criminal no Brasil. O documento pode conter diversas seções: o início com “Vistos” ou “SENTENÇA”, o relatório dos fatos, a fundamentação, a parte dispositiva (decisória) e demais informações, como depoimentos, declarações, referências a denúncias, fundamentos legais e menções à apreensão de drogas. O texto incluirá dados sobre o réu, o juiz, a vara, as penas e diversas flags e avaliações que indicam aspectos processuais específicos.

Instruções de Extração:
1. Extraia os dados para cada par processo/reu. Se na sentença houver mais de um réu, produza um objeto JSON separado para cada par, contendo todas as informações relevantes.
2. Se alguma informação não estiver presente ou não se aplicar, utilize o valor "None" (ou null) para esse campo.
3. Respeite os formatos indicados para cada campo, especialmente para quantidades numéricas, datas e textos jurídicos.
4. Extraia exatamente os campos descritos abaixo, sem omitir ou resumir nenhum detalhe.
5. Nas seções referentes a flags e avaliações (campos com prefixos "flag_" e "aval_"), não se limite à mera correspondência textual exata. Avalie se o texto da sentença descreve as circunstâncias correspondentes aos conceitos abaixo, mesmo que expressos de formas variadas. Utilize seu julgamento para identificar sinônimos, expressões equivalentes ou contextos que indiquem a presença da condição descrita..

**Regras Críticas:**
1. Campos numéricos: Sempre em gramas (apenas números)
2. Campos Sim/Não: Apenas "Sim" ou "Não"
3. Processo: Exatamente 20 dígitos
4. Flags: True apenas se explicitamente mencionado
5. Nada de markdown ou texto extra

**Schema JSON:**
{schema_json}

**Instruções Adicionais:**
- Se houver mais de um réu, gere um objeto JSON separado para cada par processo/reu.
- Se alguma informação não estiver presente, utilize "None" ou null.
- Respeite os formatos indicados para cada campo.
- Avalie o contexto para preencher flags e avaliações, mesmo que expressos de formas variadas.

<</SYS>>

## Documento:
{document}

## Saída (APENAS JSON): 
[/INST]
"""

In [4]:

def criar_extrator():
    model = models.transformers("meta-llama/Llama-3.2-3B-Instruct")
    return generate.json(model, SentencaModel)

def parse_resposta(texto: str, max_retries=3):
    print(f"Init extractor at {time.time()}")
    
    extrator = criar_extrator()
    
    print(f"Init loop at {time.time()}")
    for _ in range(max_retries):
        try:
            resposta = extrator(PROMPT_TEMPLATE.format(
                document=texto,
                schema_json=json.dumps(SentencaModel.model_json_schema(), indent=2, ensure_ascii=False)
            ))
            print(f"Ended at {time.time()}")
            return resposta.model_dump()
        except ValidationError as e:
            print(f"Erro na tentativa {_+1}: {e}")
    raise ValueError("Falha após 3 tentativas")

    

# Salvando o JSON para validação posterior
# with open("resultado.json", "w", encoding="utf-8") as f:
#     json.dump(resultado, f, ensure_ascii=False, indent=2)

In [5]:
df_teste = pd.read_parquet("validation.parquet")[0:2]

In [6]:
sentenca = df_teste["julgado"].values[0]

In [7]:
resultado = parse_resposta(sentenca)
print(resultado)

Init extractor at 1741094465.411427


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Init loop at 1741094502.4082189
Ended at 1741095326.7336779
{'processo': '00317369190718072605', 'juiz': 'Justi�.,', 'sexo_juiz': 'Masculino', 'vara': 'Justi�. de Droga e Condutas Afins', 'nome': 'GUSTAVO RODRIGUES PATR\\u00cdIO', 'local': 'Rua Jo\\u00e3o da Cunha Lobo, n\\u00e2o 26', 'maconha': 'None', 'maconha_g': None, 'cocaina': '182 pinos', 'cocaina_g': None, 'crack': None, 'crack_g': None, 'ecstasy': None, 'ecstasy_g': None, 'lsd': 'None', 'lsd_g': None, 'outras': None, 'anabolizantes': 'Sim', 'anorexigenos': 'Não', 'haxixe': 'Sim', 'skank': 'Não', 'lanca_perfume': 'Sim', 'tolueno': 'Não', 'den_drog': 'Sim', 'den_outros': None, 'sentenca': 'Procedente à pretensão punitiva para a absolvição de Gustavo Rodrigues Patrício', 'res_drogas': 'Cocaina e Maconha', 'res_outros': None, 'pena_base': 'Detenção de 1 a 3 anos, ou multa', 'agravantes33_agrup': 'Não', 'confissao': 'Não', 'menoridade': 'Não', 'atenuantes33_agrup': 'Não', 'adolescente': 'Não', 'arma_de_fogo': 'Não', 'interestadual'

In [16]:
df_teste.to_json(orient="records", force_ascii=False)




'[{"id":383.0,"processo":"00317369020178260050","julgado":"SENTENÇA Processo nº: 0031736-90.2017.8.26.0050 - Controle nº 853\\/17 Classe – Assunto: Procedimento Especial da Lei Antitóxicos - Tráfico de Drogas e Condutas Afins Autor: Justiça Pública Réu: GUSTAVO RODRIGUES PATRICIO VISTOS, etc. GUSTAVO RODRIGUES PATRICIO, qualificado nos autos, foi denunciado como incurso nas sanções do art. 33, caput, da Lei nº 11.343\\/06, porque, no dia 20 de abril de 2.017, por volta das 02:00 horas, na Rua João da Cunha Lobo, altura do nº 42, Cangaíba, nesta capital, trazia consigo, a consumo de terceiros, 182 pinos de cocaína e 24 invólucros de maconha, substâncias entorpecentes que determinam dependência física ou psíquica, sem autorização e em desacordo com determinação legal e regulamentar. O réu foi notificado para apresentar defesa prévia, a qual foi juntada aos autos por meio de defensora pública. Em seguida, a denúncia foi recebida e o réu citado. Durante a instrução processual foram ouvidas

In [ ]:
pd.read_parquet("validation.parquet").drop(columns=["julgado"]).to_json(orient="records", force_ascii=False)

'[{"id":383.0,"processo":"00317369020178260050","juiz":"AUGUSTO ANTONINI","sexo_juiz":"Masculino","vara":"28ª Vara Criminal","nome":"GUSTAVO RODRIGUES PATRICIO","local":"Via Pública","maconha":"24 porções. 50,4 g","maconha_g":50.4,"cocaina":"181 porções. 100,4 g","cocaina_g":100.4,"crack":"0","crack_g":0.0,"ecstasy":"0","ecstasy_g":0.0,"lsd":"0","lsd_g":0.0,"outras":"0","anabolizantes":"Não","anorexigenos":"Não","haxixe":"Não","skank":"Não","lanca_perfume":"Não","tolueno":"Não","den_drog":"Art. 33","den_outros":"Não","sentenca":"Absolvição","res_drogas":"NA","res_outros":"NA","pena_base":"NA","agravantes33_agrup":"NA","confissao":"Não","menoridade":"Não","atenuantes33_agrup":"NA","adolescente":"Não","arma_de_fogo":"Não","interestadual":"Não","concurso_formal":"Não","estabelecimento":"Não","aumento33_agrup":"NA","paragrafo_4o_agrupado":"Não cabimento \\/ Não informado","pena33":"NA","pena33_meses":0.0,"pena_drogas":"NA","pena_outros":"NA","tot_pen":"NA","tot_pen_meses":0.0,"substituicao

In [18]:
df = pd.read_parquet("validation.parquet").drop(columns=["julgado"])

In [19]:
df.describe()


,id,maconha_g,cocaina_g,crack_g,ecstasy_g,lsd_g,pena33_meses,tot_pen_meses
count,51.000000,51.000000,51.000000,51.000000,51.000000,51.000000,51.000000,51.000000
mean,206.588235,7021.890196,366.058824,49.228431,4.829412,0.007843,42.718954,52.169935
std,135.899989,31323.986493,879.205971,278.369399,34.488898,0.033723,38.351032,63.815929
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,96.500000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000,20.000000
50%,189.000000,25.200000,16.600000,0.000000,0.000000,0.000000,30.000000,30.000000
75%,345.500000,135.250000,86.850000,4.250000,0.000000,0.000000,66.500000,70.000000
max,416.000000,209300.000000,3609.200000,1961.400000,246.300000,0.200000,180.000000,372.000000


In [22]:
# contar os valores unicos de cada coluna
for col in df.columns:
    # se o valor for menor que 10, listar os valores
    if df[col].nunique() < 10:
        print(f"{col}: {df[col].unique()}")
    else:
        print(f"{col}: {df[col].nunique()} valores únicos")

id: 51 valores únicos
processo: 48 valores únicos
juiz: 37 valores únicos
sexo_juiz: ['Masculino' 'Feminino']
vara: 25 valores únicos
nome: 51 valores únicos
local: ['Via Pública' 'Residência' 'Área não ocupada' 'Comércio e serviços'
 'Restaurante e afins' 'Repartição Pública' 'Terminal/Estação']
maconha: 34 valores únicos
maconha_g: 34 valores únicos
cocaina: 34 valores únicos
cocaina_g: 33 valores únicos
crack: 18 valores únicos
crack_g: 18 valores únicos
ecstasy: ['0' '401 comprimidos. 246,3 g']
ecstasy_g: [  0.  246.3]
lsd: ['0' '5 porções. 0,1 g' '13 porções. 0,2 g']
lsd_g: [0.  0.1 0.2]
outras: ['0' 'Lança-perfume. 93 frascos. 2790 ml' 'Haxixe. 5 porções. 3,2 g'
 'Anabolizantes. Anorexígenos']
anabolizantes: ['Não' 'Sim']
anorexigenos: ['Não' 'Sim']
haxixe: ['Não' 'Sim']
skank: ['Não']
lanca_perfume: ['Não' 'Sim']
tolueno: ['Não']
den_drog: ['Art. 33' 'Art. 33. Art. 35' 'Art. 33. Art. 34. Art. 35'
 'Art. 33. Art. 34']
den_outros: ['Não' 'Art. 329 CP' 'Art. 12 L10826. Art. 16 L108